In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import joblib


In [ ]:
df = pd.read_csv("../data/cardiovascular.csv")
print("Dataset Preview:")
print(df.head())
print("\nColumns:", list(df.columns))


In [ ]:
# Binary target: High cardiovascular risk vs not high
y = (df["risk_category"] == "High").astype(int)

# Drop non-feature columns
X = df.drop(columns=["Patient_ID", "heart_disease_risk_score", "risk_category"]).copy()

# Handle column name safely
family_col = "family_history_heart_disease" if "family_history_heart_disease" in X.columns else "family_history"

categorical_mappings = {
    "smoking_status": {"Never": 0, "Former": 1, "Current": 2},
    family_col: {"No": 0, "Yes": 1},
}

for col, mapping in categorical_mappings.items():
    X[col] = X[col].map(mapping)

X = X.fillna(0)
feature_columns = X.columns.tolist()

print("Features used:", feature_columns)
print("Target balance (High risk=1):")
print(y.value_counts(normalize=True))


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
selector = SelectKBest(score_func=f_classif, k=min(10, X_train.shape[1]))

X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_test_selected = selector.transform(X_test_scaled)


In [ ]:
lr_model = LogisticRegression(max_iter=2000, random_state=42)
rf_model = RandomForestClassifier(n_estimators=300, random_state=42)

lr_model.fit(X_train_selected, y_train)
rf_model.fit(X_train_selected, y_train)


def evaluate_model(model, X_eval, y_eval):
    y_pred = model.predict(X_eval)
    return {
        "Accuracy": accuracy_score(y_eval, y_pred),
        "Precision": precision_score(y_eval, y_pred, zero_division=0),
        "Recall": recall_score(y_eval, y_pred, zero_division=0),
        "F1": f1_score(y_eval, y_pred, zero_division=0),
    }


lr_results = evaluate_model(lr_model, X_test_selected, y_test)
rf_results = evaluate_model(rf_model, X_test_selected, y_test)

print("\nLogistic Regression:", lr_results)
print("Random Forest:", rf_results)

best_model = rf_model if rf_results["F1"] > lr_results["F1"] else lr_model
print("\nBest Model Selected:", "Random Forest" if best_model == rf_model else "Logistic Regression")


In [ ]:
final_model = best_model

best_threshold = 0.5
best_f1 = 0

y_probs = final_model.predict_proba(X_test_selected)[:, 1]

for thresh in np.arange(0.2, 0.91, 0.02):
    y_pred_thresh = (y_probs >= thresh).astype(int)
    f1 = f1_score(y_test, y_pred_thresh, zero_division=0)

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = float(round(thresh, 2))

print("\nBest Threshold:", best_threshold)
print("Best F1 Score:", best_f1)

y_final_pred = (y_probs >= best_threshold).astype(int)

print("\nFinal Evaluation:")
print("Accuracy:", accuracy_score(y_test, y_final_pred))
print("Precision:", precision_score(y_test, y_final_pred, zero_division=0))
print("Recall:", recall_score(y_test, y_final_pred, zero_division=0))
print("F1 Score:", f1_score(y_test, y_final_pred, zero_division=0))


In [ ]:
joblib.dump({
    "model": final_model,
    "scaler": scaler,
    "selector": selector,
    "threshold": best_threshold,
    "feature_columns": feature_columns,
    "categorical_mappings": categorical_mappings,
    "positive_label": "High Cardiovascular Risk",
    "negative_label": "Not High Cardiovascular Risk",
    "target_definition": "risk_category == 'High'",
    "source_dataset": "../data/cardiovascular.csv",
}, "../models/heart_model.pkl")

print("\nModel saved successfully!")
